# Windowing and the signal-to-noise ratio

Taking the Fourier transform of a finite chunk of data assumes it repeats forever, and **windowing** (tapering the data to zero at the edges before transforming it) is the fix. This version spends a bit more time being precise about why that works and what it costs.

We'll see spectral leakage in a toy example, explain what we can do to limit it and what it costs.

## Setup

In [ ]:
import warnings

warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")   # a harmless warning from LAL

import matplotlib.mlab as mlab
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal.windows import hann, tukey

import scipy.interpolate
from scipy.signal import welch


from gwosc.datasets import event_gps
from gwpy.timeseries import TimeSeries
from scipy.interpolate import interp1d

%config InlineBackend.figure_format = 'retina'

## Why finite data leaks

The DFT treats our \(N\) samples as one period of a signal repeating forever. If the data doesn't complete a whole number of cycles, there is a jump at the wrap-around point where one repeat meets the next.

Mathematically, if the signal is not periodic over the observation interval \(T\), then the endpoints do not match:

$$
x(0)\neq x(T).
$$

When the DFT periodically repeats the segment, this creates a discontinuity

$$
\Delta x=x(0)-x(T).
$$

A discontinuity is a sharp feature, and a sharp feature cannot be represented by a single sinusoid. It requires many Fourier components, which is the same reason a square wave needs many harmonics to reproduce its sharp edges. The energy therefore spreads into other frequency bins:

$$
\boxed{
\text{non-periodicity}
\;\Rightarrow\;
\text{edge discontinuity}
\;\Rightarrow\;
\text{many Fourier components}
\;\Rightarrow\;
\text{spectral leakage}
}
$$

For a sinusoid,

$$
x(t)=\sin(2\pi f_0t),
$$

the endpoints match when $f_0T\in\mathbb{Z}$, meaning the signal completes an integer number of cycles. For example, a 50 Hz signal observed for 1 s completes exactly 50 cycles, so there is no jump. A 50.5 Hz signal completes 50.5 cycles, so the endpoints do not match and leakage appears.

In [ ]:
fs = 1000
t = np.arange(0, 1, 1 / fs)

f = np.fft.rfftfreq(len(t), 1 / fs)

def spectrum(f0):
    x = np.sin(2 * np.pi * f0 * t)
    X = np.fft.rfft(x)
    amp = 2 * np.abs(X) / len(x)
    return x, amp

cases = [
    ("50 Hz", 50),
    ("50.5 Hz", 50.5),
]

fig, axes = plt.subplots(2, 2, figsize=(10, 6))

for col, (label, f0) in enumerate(cases):
    x, amp = spectrum(f0)

    # Time domain
    axes[0, col].plot(t, x)
    axes[0, col].set_title(label)
    axes[0, col].set_xlabel("time (s)")
    axes[0, col].set_ylabel("amplitude")
    axes[0, col].grid(alpha=0.3)

    # Frequency domain
    axes[1, col].semilogy(f, amp)
    axes[1, col].set_xlim(0, 150)
    axes[1, col].set_ylim(1e-5, 2)
    axes[1, col].set_xlabel("frequency (Hz)")
    axes[1, col].set_ylabel("amplitude")
    axes[1, col].grid(alpha=0.3, which="both")

plt.tight_layout()
plt.show()

# Leakage between 10 and 40 Hz
band = (f > 10) & (f < 40)

for label, f0 in cases:
    _, amp = spectrum(f0)
    print(f"{label:10s}  leakage 10-40 Hz = {amp[band].max():.5f}")

### Why do we taper?

Instead of abruptly cutting the data from \(1\) to \(0\), we multiply it by a smooth window \(w(t)\) that gradually goes to zero at both edges:

$$
x_{\mathrm{windowed}}(t)=x(t)w(t),
\qquad
w(0)=w(T)=0.
$$

Now the periodic extension joins smoothly:

$$
\boxed{x_{\mathrm{windowed}}(0)=x_{\mathrm{windowed}}(T)=0}.
$$

The large boundary jump is therefore removed. In frequency space, multiplication by the window corresponds to convolution:

$$
X_{\mathrm{windowed}}(f)
=
X(f)*W(f).
$$

A smooth taper has much smaller sidelobes in $W(f)$ than the abrupt rectangular cutoff. The convolution therefore puts much less energy into frequencies far away from $f_0$, reducing spectral leakage.

The trade-off is that the main lobe becomes wider: this means lower frequency resolution.

In [ ]:
# Hann window
w = hann(len(t), sym=False)
x_windowed = x * w

# FFTs
X = np.fft.rfft(x)
X_windowed = np.fft.rfft(x_windowed)

# Amplitudes
amp = 2 * np.abs(X) / len(x)
amp_windowed = 2 * np.abs(X_windowed) / w.sum()

fig, axes = plt.subplots(2, 1, figsize=(10, 7))

# Time domain
axes[0].plot(t, x, label="Not windowed")
axes[0].plot(t, x_windowed, label="Hann windowed")
axes[0].set_xlabel("time (s)")
axes[0].set_ylabel("amplitude")
axes[0].set_title("Time domain")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Frequency domain
axes[1].semilogy(f, amp, label="Not windowed")
axes[1].semilogy(f, amp_windowed, label="Hann windowed")
axes[1].set_xlim(0, 150)
axes[1].set_ylim(1e-5, 2)
axes[1].set_xlabel("frequency (Hz)")
axes[1].set_ylabel("amplitude")
axes[1].set_title("Frequency domain")
axes[1].legend()
axes[1].grid(alpha=0.3, which="both")

plt.tight_layout()
plt.show()

band = (f > 10) & (f < 40)

leakage = amp[band].max()
leakage_windowed = amp_windowed[band].max()

print(f"Leakage 10-40 Hz (not windowed):  {leakage:.5f}")
print(f"Leakage 10-40 Hz (Hann windowed): {leakage_windowed:.5f}")

A few points worth being precise about:

* **The cost is frequency resolution.** A window that decays faster in its sidelobes always has a wider main peak, because forcing the edges to zero is, in effect, keeping less of the data. Leakage vs. resolution is a trade-off.

* **A window changes the transform's overall scale** too, since it multiplies the data down before transforming; dividing by `w.sum()` above corrects for that, so the reported amplitudes stay honest.

A common compromise is the **Tukey window**: flat and equal to one over most of the segment, tapered only at the two edges (a fraction $\alpha$ of the segment is tapered; $\alpha=0$ is no window, $\alpha=1$ is Hann).

The **Hann window** is

$$
w[n] = \frac{1}{2}\left(1-\cos\left(\frac{2\pi n}{N}\right)\right),
\qquad 0 \le n < N.
$$

The **Tukey window** can be written as

$$
w[n] =
\begin{cases}
\frac{1}{2}\left[1+\cos\left(\pi\left(\frac{2n}{\alpha(N-1)}-1\right)\right)\right],
& 0 \le n < \frac{\alpha(N-1)}{2}, \\[6pt]
1,
& \frac{\alpha(N-1)}{2} \le n \le (N-1)\left(1-\frac{\alpha}{2}\right), \\[6pt]
\frac{1}{2}\left[1+\cos\left(\pi\left(\frac{2n}{\alpha(N-1)}-\frac{2}{\alpha}+1\right)\right)\right],
& (N-1)\left(1-\frac{\alpha}{2}\right) < n < N.
\end{cases}
$$

Thus,

$$
\alpha=0 \quad\Rightarrow\quad \text{rectangular (no) window},
$$

and

$$
\alpha=1 \quad\Rightarrow\quad \text{Hann window}.
$$

For our example, $\alpha=0.2$, so only the outer 20% of the segment is tapered, with roughly 10% tapered at each end. 

In [ ]:
# Windows
windows = {
    "No window": np.ones(len(t)),
    "Tukey": tukey(len(t), alpha=0.2, sym=False),
    "Hann": hann(len(t), sym=False),
}

fig, axes = plt.subplots(2, 1, figsize=(10, 7))

for name, w in windows.items():
    x_windowed = x * w
    axes[0].plot(t, x_windowed, label=name)

axes[0].set_xlim(0, 1)
axes[0].set_xlabel("time (s)")
axes[0].set_ylabel("amplitude")
axes[0].set_title("Time domain")
axes[0].legend(loc="upper right")
axes[0].grid(alpha=0.3)

for name, w in windows.items():
    X = np.fft.rfft(x * w)
    amp = 2 * np.abs(X) / w.sum()

    axes[1].semilogy(f, amp, label=name)

axes[1].set_ylim(1e-5, 2)
axes[1].set_xlabel("frequency (Hz)")
axes[1].set_ylabel("amplitude")
axes[1].set_title("Frequency domain")
axes[1].legend()
axes[1].grid(alpha=0.3, which="both")

plt.tight_layout()
plt.show()

band = (f > 10) & (f < 40)

print("Leakage in 10–40 Hz")
print("-" * 35)

for name, w in windows.items():
    X = np.fft.rfft(x * w)
    amp = 2 * np.abs(X) / w.sum()

    leakage = amp[band].max()
    print(f"{name:15s}: {leakage:.6f}")

# Signal to Noise Ratio

We can ask what is the overlap of our time domain data $d$ with a specific gravitational wave waveform $h$  $$\rho = \frac{(d|h)}{\sqrt{(h|h)}}.$$ $\rho$ is called the **matched-filter signal to noise ratio**. It is big when the data lines up with $h$ and that agreement occurs in the quiet parts of the detector band.

The noise weighted inner product between two time series a and b is defined as $$(a|b) \equiv 4\, \mathrm{Re} \int_{0}^{\infty} \frac{\tilde{a}(f)\,\tilde{b}^{*}(f)}{S_n(f)}\, df \: \longrightarrow 4\, \mathrm{Re} \sum_{f \in (0,...,f_{\text{max}})} \frac{\tilde{a}(f)\,\tilde{b}^{*}(f)}{S_n(f)}\, \Delta f$$ where  $S_n(f)$ is the one-sided PSD of the noise.

When we plug in the waveform $h$ itsself into this formula, we obtain the **optimal SNR** 


$$\rho^2 = 4\int_0^\infty \frac{|\tilde h(f)|^2}{S_n(f)}\,df,$$

Notice that $1/S_n(f)$ is also the weighting used for whitening.

## Task: The SNR of GW250114 - The Loudest Black Hole Binary Signal in LIGO

https://gwosc.org/eventapi/html/GWTC-5.0/GW250114_082203/v2/

https://journals.aps.org/prl/pdf/10.1103/kw5g-d732 - Used for tests of GR!

Source parameters:
- $m_1 = 33.56 \text{ M}_{\odot}$
- $m_2 = 32.0 \text{ M}_{\odot}$
- distance = 387 Mpc

In [ ]:
h1_ts = TimeSeries.read("data/GW250114_H1_strain.hdf5")
tevent = 1420878141.2      # GPS time of GW250114, event_gps("GW250114_082203")
gpstime = h1_ts.times.value
h1strain = h1_ts.value
t = gpstime - tevent

In [ ]:
fs = 4096                    # sampling rate (Hz)
dt = 1 / fs                  # time between samples (s)
N = len(h1strain)            # number of samples
duration = N * dt            # total duration (s)
print(f"N = {N} samples, dt = {dt * 1e3:.4f} ms, duration = {duration:.0f} s")

In [ ]:
bandpass = (20, 300)                   # Hz: the band we keep
h1_gwpy_white = h1_ts.whiten(4, 2).bandpass(*bandpass)
around_gwpy = h1_gwpy_white.crop(tevent - 0.15, tevent + 0.15)

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(around_gwpy.times.value - tevent, around_gwpy.value)
ax.set_xlabel("time relative to coalesence (s)")
ax.set_ylabel("whitened strain")
ax.set_title("Whitened H1 strain around GW250114")
plt.show()

### TO DO: compute the optimal SNR of the event

TO DO: Complete the snr function. As you write it, think about which quantities you need and how you would compute them from the time domain gravitational wave strain. 

$$\rho^2 = 4\int_0^\infty \frac{|\tilde h(f)|^2}{S_n(f)}\,df,$$

In [ ]:
# -- TO DO: Complete the optimal SNR function 

def snr(d,h,psd,f):
    """
    d,h : frequency domain data to go into the inner product
    f: frequencies at which h,d and psd are evaluated
    psd: psd of the noise
    """

    return ...

## The noise PSD

We compute the noise PSD from the part of the data that doesn't include a GW signal.

In [ ]:
exclude_half_width = 0.5   # seconds around the event to exclude — adjust based on the plot 
psd_crop = np.abs(t) > exclude_half_width   # True everywhere except near the merger
psd_freqs, psd_welch = welch(h1_ts.value[psd_crop], fs=fs,nperseg=int(4 * fs), average="median")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))
ax1.plot(t, h1_gwpy_white, color="#4a3aa7", linewidth=0.5)
ax1.axvspan(-0.5, 0.5, color="red", alpha=0.2, label="excluded region (around merger)")
ax1.set_xlabel("Time from event (s)")
ax1.set_ylabel("whitened strain")
ax1.set_title("H1 strain — full segment, signal region shaded")
ax1.legend()
ax2.loglog(psd_freqs, psd_welch, color="#4a3aa7", label="Welch PSD (off-source)")
ax2.set_xlabel("Frequency [Hz]")
ax2.set_ylabel("PSD [1/Hz]")
ax2.set_title("H1 noise PSD")
ax2.grid(True, which="both", alpha=0.3)
ax2.legend()
plt.tight_layout()
plt.show()

## The waveform

In [ ]:
signal_data = np.load("data/GW250114_H1_signal.npz")
h = signal_data["h1_signal"]
h_freqs = signal_data["freqs"]

In [ ]:
plt.loglog(h_freqs,np.abs(h),color="#4a3aa7")
plt.ylabel("|h(f)|")
plt.xlabel("Frequency [Hz]")
plt.xlim(left = 10)
plt.show()

## Finally the optimal SNR

In [ ]:
snr(h,h,psd_welch,h_freqs)

## Ups... Why does this not work yet?

We still need to interpolate the psd and evaluate it at the frequency grid of the waveform. 

In [ ]:
interp_psd = interp1d(psd_freqs, psd_welch, bounds_error=False, fill_value=np.inf)
psd_on_waveform_grid = interp_psd(h_freqs)

In [ ]:
optimal_snr = snr(h,h,psd_on_waveform_grid,h_freqs)
print("Optimal SNR of GW250114 signal in Hanford =" ,optimal_snr)

And this is only the SNR given the data from one detector! 